[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iamjamesbowden/AG952/blob/main/materials/week07/notebooks/AG952_Week07_Workshop3_StudentNotebook.ipynb)

## AG952 | Workshop 3: The Analyst's Brief

### Decision-gated textual analysis of 10-K filings

*Dr James Bowden, Strathclyde Business School*

---

Your team is a small analyst group at a textual research desk. A senior colleague has flagged five firms whose 10-K filings preceded a significant stock price move. The desk needs to determine whether the language in those filings was signalling something the market missed. Your team will build a textual evidence file for your assigned firm, making and justifying every methodological choice along the way.

**Run each cell in order using Shift+Enter. Read the feedback at each checkpoint before proceeding to the next.**

**Total time: 60 minutes.**

In [ ]:
#@title Setup -- run this cell first (takes 2-3 minutes)
!pip install sec-edgar-downloader ipywidgets nltk spacy textstat \
            scikit-learn wordcloud matplotlib seaborn pandas \
            requests beautifulsoup4 -q

import re, time, json, warnings
from collections import Counter
from IPython.display import display, HTML, Markdown, clear_output
import ipywidgets as widgets
import requests
from bs4 import BeautifulSoup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import textstat
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
warnings.filterwarnings("ignore")

for _pkg, _path in [("punkt_tab","tokenizers/punkt_tab"),
                    ("stopwords","corpora/stopwords"),
                    ("wordnet","corpora/wordnet"),
                    ("averaged_perceptron_tagger","taggers/averaged_perceptron_tagger")]:
    try: nltk.data.find(_path)
    except LookupError: nltk.download(_pkg, quiet=True)

try:
    import spacy
    try: _nlp = spacy.load("en_core_web_md")
    except OSError:
        import subprocess
        subprocess.run(["python","-m","spacy","download","en_core_web_md"], check=True, capture_output=True)
        _nlp = spacy.load("en_core_web_md")
    _SPACY_OK = True
except Exception as _e:
    _SPACY_OK = False
    print(f"spaCy unavailable ({_e}). Lemmatisation will use a rule-based fallback.")

_stemmer = PorterStemmer()
print("Setup complete.")


In [ ]:
#@title Constants -- do not edit

REPO_RAW   = "https://raw.githubusercontent.com/iamjamesbowden/AG952/main"
DATA_BASE  = f"{REPO_RAW}/materials/week07/data"
CACHE_BASE = f"{DATA_BASE}/10k_cache"

LM_NEG_URL      = f"{DATA_BASE}/lm_negative.txt"
LM_POS_URL      = f"{DATA_BASE}/lm_positive.txt"
LM_UNC_URL      = f"{DATA_BASE}/lm_uncertainty.txt"
HARV_NEG_URL    = f"{DATA_BASE}/harvard_iv4_negative.txt"
HARV_POS_URL    = f"{DATA_BASE}/harvard_iv4_positive.txt"
FINANCIAL_URL   = f"{DATA_BASE}/financial_context.csv"

SPREADSHEET_ID  = "1UrgwAXYkJabGAsfI4uxAdTEKkHE5deHUbk7yxvF9DBg"
WORKSHEET_NAME  = "Workshop3"

EDGAR_HEADERS   = {"User-Agent": "AG952 Strathclyde University ag952course@strath.ac.uk"}
EDGAR_DATA      = "https://data.sec.gov"
EDGAR_ARCHIVES  = "https://www.sec.gov/Archives/edgar/data"

FIRMS = {
    "Boeing": {
        "cik": "0000012927",
        "years": [2017, 2018, 2019],
        "slug": "boeing",
        "note": ("Boeing's 2019 10-K was filed in the shadow of two fatal 737 MAX crashes and a "
                 "global grounding. Consider what changed in the language of the Risk Factors "
                 "section between 2018 and 2019.")
    },
    "Intel": {
        "cik": "0000050863",
        "years": [2021, 2022, 2023],
        "slug": "intel",
        "note": ("Intel's filings across 2021-2023 span a period of aggressive restructuring "
                 "rhetoric and simultaneous margin compression. Consider whether the language "
                 "tracks the financial reality.")
    },
    "Peloton": {
        "cik": "0001639825",
        "years": [2020, 2021, 2022],
        "slug": "peloton",
        "note": ("Peloton went from pandemic darling to crisis management between 2020 and 2022. "
                 "The MD&A sections across these years tell a story -- your task is to measure it.")
    },
    "SVB Financial": {
        "cik": "0000719739",
        "years": [2020, 2021, 2022],
        "slug": "svb_financial",
        "note": ("SVB Financial's 2022 10-K was filed four months before the bank's collapse. "
                 "Consider what a textual analyst reading that filing would have seen.")
    },
}

GSHEET_COLUMNS = [
    "team_name","firm","cp1_year","cp1_section","cp2_normalisation","cp2_stopwords",
    "cp2_remove_numbers","cp2_lowercase","cp3_dictionary","cp3_denominator",
    "cp3_lm_net","cp3_harv_net","cp4_year_mode","cp4_frame","cp4_fog",
    "cp5_pair","cp5_tf_similarity","cp5_tfidf_similarity","cp6_analyst_note"
]
print("Constants loaded.")


In [ ]:
#@title INSTRUCTOR CELL -- set team assignments before session
# Map each team name to a firm from: Boeing, Intel, Peloton, SVB Financial
# Ensure at least one team per firm. Edit this cell only.

TEAM_ASSIGNMENT = {
    "The Write-Offs":            "Boeing",
    "Item 1A-OK":                "Boeing",
    "The Going Concerns":        "Boeing",
    "The Risk Factors":          "Boeing",
    "The False Negatives":       "Intel",
    "The Obfuscators":           "Intel",
    "MD&Aye":                    "Intel",
    "The Uncertainty Principle": "Intel",
    "Standard Deviants":         "Peloton",
    "The Boilerplate Boys":      "Peloton",
    "Too Big to Fog":            "Peloton",
    "The Negative Sentiments":   "Peloton",
    "The Material Weaknesses":   "SVB Financial",
    "The Hedgers":               "SVB Financial",
    "Off Balance Sheet":         "SVB Financial",
}
print("Team assignments set.")


In [ ]:
#@title Instructor configuration -- paste Web App URL before session
# Update APPS_SCRIPT_URL with the Google Apps Script Web App URL before the
# workshop. See APPS_SCRIPT_SETUP.md in the repository for deployment
# instructions. This is the only cell that needs editing before each session.

APPS_SCRIPT_URL = "[PASTE YOUR WEB APP URL HERE — see APPS_SCRIPT_SETUP.md]"
print("Apps Script URL configured." if "[PASTE" not in APPS_SCRIPT_URL
      else "WARNING: APPS_SCRIPT_URL is not set. Update this cell before the session.")


In [ ]:
#@title Session initialisation -- run once

session_data = {col: None for col in GSHEET_COLUMNS}

def validate_checkpoint(required_vars, checkpoint_name):
    missing = [v for v in required_vars if v not in globals() or globals()[v] is None]
    if missing:
        raise ValueError(
            f"Complete {checkpoint_name} before running this cell. "
            f"Missing: {', '.join(missing)}"
        )

def fetch_wordlist(url):
    """Load a plain-text wordlist from a URL, one word per line."""
    try:
        r = requests.get(url, headers=EDGAR_HEADERS, timeout=10)
        r.raise_for_status()
        return set(w.strip().lower() for w in r.text.splitlines() if w.strip())
    except Exception as e:
        print(f"Wordlist load failed ({url}): {e}")
        return set()

def load_lm_dictionary():
    """Load L&M word lists from repo-hosted files (neg, pos, uncertainty)."""
    neg = fetch_wordlist(LM_NEG_URL)
    pos = fetch_wordlist(LM_POS_URL)
    unc = fetch_wordlist(LM_UNC_URL)
    if neg or pos:
        print(f"L&M dictionary loaded: {len(neg):,} negative, {len(pos):,} positive, "
              f"{len(unc):,} uncertainty words.")
    else:
        print("L&M dictionary could not be loaded. Check your connection.")
    return neg, pos, unc

def _draw_pipeline_diagram():
    """Render a simple horizontal pipeline diagram for the six checkpoints."""
    import matplotlib.patches as mpatches
    import matplotlib.pyplot as plt
    stages = [
        ("CP0", "Team\nRegistration"),
        ("CP1", "Data\nSourcing"),
        ("CP2", "Pre-\nprocessing"),
        ("CP3", "Sentiment\nAnalysis"),
        ("CP4", "Readability\nAnalysis"),
        ("CP5", "Cosine\nSimilarity"),
        ("CP6", "Analyst\nNote"),
    ]
    colours = ["#4f46e5","#0ea5e9","#10b981","#f59e0b","#ef4444","#8b5cf6","#06b6d4"]
    fig, ax = plt.subplots(figsize=(13, 2.4))
    ax.set_xlim(0, len(stages))
    ax.set_ylim(0, 1)
    ax.axis("off")
    bw, bh, by = 0.82, 0.56, 0.22
    for i, ((code, label), col) in enumerate(zip(stages, colours)):
        cx = i + 0.5
        rect = mpatches.FancyBboxPatch(
            (cx - bw/2, by), bw, bh,
            boxstyle="round,pad=0.04", linewidth=1.2,
            edgecolor="white", facecolor=col, zorder=3)
        ax.add_patch(rect)
        ax.text(cx, by + bh/2 + 0.08, code, ha="center", va="center",
                fontsize=8, fontweight="bold", color="white", zorder=4)
        ax.text(cx, by + bh/2 - 0.10, label, ha="center", va="center",
                fontsize=6.5, color="white", zorder=4, linespacing=1.3)
        if i < len(stages) - 1:
            ax.annotate("", xy=(i + 1 + 0.5 - bw/2 - 0.02, by + bh/2),
                        xytext=(cx + bw/2 + 0.02, by + bh/2),
                        arrowprops=dict(arrowstyle="->", color="#64748b", lw=1.4),
                        zorder=2)
    ax.set_title("Workshop 3 pipeline -- run checkpoints in order",
                 fontsize=9, color="#475569", pad=6)
    plt.tight_layout()
    plt.show()

def fetch_filing_text(cik, year, headers=EDGAR_HEADERS, max_chars=700000):
    """Fetch the primary 10-K document from EDGAR REST API."""
    cik_p = str(cik).lstrip("0").zfill(10)
    url = f"{EDGAR_DATA}/submissions/CIK{cik_p}.json"
    try:
        r = requests.get(url, headers=headers, timeout=15)
        r.raise_for_status()
        data = r.json()
        time.sleep(0.3)
        recent = data["filings"]["recent"]
        for i, form in enumerate(recent["form"]):
            if form == "10-K" and int(recent["filingDate"][i][:4]) == year:
                acc = recent["accessionNumber"][i].replace("-", "")
                doc = recent["primaryDocument"][i]
                cik_int = str(int(cik.lstrip("0")))
                filing_url = f"{EDGAR_ARCHIVES}/{cik_int}/{acc}/{doc}"
                r2 = requests.get(filing_url, headers=headers, timeout=40)
                r2.raise_for_status()
                time.sleep(0.3)
                soup = BeautifulSoup(r2.text, "html.parser")
                for tag in soup(["table","script","style","ix:header","ix:nonfraction"]):
                    tag.decompose()
                text = soup.get_text(separator=" ")
                return re.sub(r"\s+", " ", text).strip()[:max_chars]
    except Exception as e:
        print(f"EDGAR fetch failed ({cik}, {year}): {e}")
    return None

def fetch_cached_text(slug, year, section_key):
    """Load cached section text from AG952 GitHub repo."""
    fname = f"{slug}_{year}_{section_key}.txt"
    url = f"{CACHE_BASE}/{fname}"
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            return r.text
    except Exception:
        pass
    return None

def extract_section(full_text, section):
    """Extract Item 1A or Item 7 from full 10-K text using regex."""
    if "Item 1A" in section:
        pats = [
            r"(?:ITEM|Item)\s+1A[\s.]+.*?(?=(?:ITEM|Item)\s+[12][^A0-9])",
        ]
    else:
        pats = [
            r"(?:ITEM|Item)\s+7[^A0-9].*?(?=(?:ITEM|Item)\s+7A|(?:ITEM|Item)\s+8)",
        ]
    for pat in pats:
        m = re.search(pat, full_text, re.DOTALL | re.IGNORECASE)
        if m and len(m.group().split()) >= 300:
            return m.group()
    return None

def apply_pipeline(text, normalisation, stopword_choice, remove_numbers, lowercase):
    """Apply pre-processing pipeline; return filtered token list."""
    if lowercase:
        text = text.lower()
    # Tokenise
    if normalisation == "Lemmatisation (spaCy en_core_web_md)" and _SPACY_OK:
        doc = _nlp(text[:200000])
        tokens = [t.lemma_.lower() for t in doc if t.is_alpha]
    elif normalisation == "Porter Stemmer":
        raw = re.findall(r"[a-zA-Z]+", text)
        tokens = [_stemmer.stem(t.lower()) for t in raw]
    else:
        tokens = [t.lower() for t in re.findall(r"[a-zA-Z]+", text)]
    # Remove numbers (mixed alphanum)
    if remove_numbers:
        tokens = [t for t in tokens if t.isalpha()]
    # Stopword filter
    MODALS = {"will","may","might","could","should","would","must","shall","can"}
    NEGATION = {"not","no","nor","never","neither","without","cannot"}
    if stopword_choice == "NLTK standard":
        sw = set(stopwords.words("english"))
        tokens = [t for t in tokens if t not in sw]
    elif stopword_choice == "L&M-aware custom list":
        sw = set(stopwords.words("english")) - MODALS - NEGATION
        tokens = [t for t in tokens if t not in sw]
    return tokens

def score_sentiment(tokens, neg_words, pos_words, denominator_tokens):
    """Score sentiment against a dictionary; return (net, matched_neg, matched_pos)."""
    m_neg = [t for t in tokens if t in neg_words]
    m_pos = [t for t in tokens if t in pos_words]
    denom = len(denominator_tokens) if denominator_tokens else len(tokens)
    net = (len(m_pos) - len(m_neg)) / denom if denom > 0 else 0.0
    return net, m_neg, m_pos

print("Session initialised. Ready to begin.")


---

## Checkpoint 0: Team Registration and Briefing

*Lecture connection: course introduction*  |  *Estimated time: 5 minutes*

Enter your team name below. Your firm assignment will be displayed automatically from the master assignment cell set by your instructor.

In [ ]:
#@title Checkpoint 0 -- Team Registration

_cp0_team_w = widgets.Dropdown(
    options=["-- select your team --"] + sorted(TEAM_ASSIGNMENT.keys()),
    description="Team name:",
    layout=widgets.Layout(width="400px")
)
_cp0_out = widgets.Output()
_cp0_btn = widgets.Button(description="Register Team", button_style="primary",
                          layout=widgets.Layout(width="160px"))

def _run_cp0(b):
    with _cp0_out:
        clear_output(wait=True)
        team = _cp0_team_w.value
        if not team or team == "-- select your team --":
            print("Please select your team name from the dropdown.")
            return
        firm = TEAM_ASSIGNMENT.get(team)
        if not firm:
            known = list(TEAM_ASSIGNMENT.keys())
            print(f"Team name not found. Registered teams: {known}")
            return
        firm_data = FIRMS[firm]
        session_data["team_name"] = team
        session_data["firm"] = firm
        globals()["_cp0_done"] = True
        globals()["_assigned_firm"] = firm
        globals()["_assigned_cik"] = firm_data["cik"]
        globals()["_assigned_years"] = firm_data["years"]
        globals()["_assigned_slug"] = firm_data["slug"]
        print(f"Team registered: {team}")
        print(f"Assigned firm  : {firm}")
        print(f"Available years: {firm_data['years']}")
        print()
        print("Context note:")
        print(firm_data["note"])
        print()
        _draw_pipeline_diagram()

_cp0_btn.on_click(_run_cp0)
display(widgets.VBox([_cp0_team_w, _cp0_btn, _cp0_out]))


---

## Checkpoint 1: Data Sourcing

*Lecture connection: Week 6, Sections 6.2 and 6.3*  |  *Estimated time: 8 minutes*

Choose which section of the 10-K to analyse. Your firm's most recent available filing
year is loaded automatically -- this is the year where the context note above applies
most directly. The notebook will attempt a live pull from SEC EDGAR and fall back to a
cached version if needed. The section choice is the substantive methodological decision
here; record your rationale, as you will need it for the analyst's note in Checkpoint 6.


In [ ]:
#@title Checkpoint 1 -- Data Sourcing

validate_checkpoint(["_cp0_done"], "Checkpoint 0")

_cp1_section_w = widgets.Dropdown(
    description="Section:",
    options=["Item 1A (Risk Factors)", "Item 7 (MD&A)", "Both"],
    layout=widgets.Layout(width="320px")
)
_cp1_out = widgets.Output()
_cp1_btn = widgets.Button(description="Run Checkpoint 1", button_style="primary",
                          layout=widgets.Layout(width="190px"))

def _run_cp1(b):
    global _cp1_done, _cp1_text, _cp1_text_comp, _cp1_year, _cp1_section
    with _cp1_out:
        clear_output(wait=True)
        year    = _assigned_years[-1]   # always use the most recent year
        section = _cp1_section_w.value
        cik     = _assigned_cik
        slug    = _assigned_slug

        def _get_section_text(sec, yr):
            """Fetch one section: EDGAR first, then cache."""
            sec_key = "item_1a" if "1A" in sec else "item_7"
            print(f"  Fetching {sec} ({yr}) from EDGAR...", end=" ", flush=True)
            full = fetch_filing_text(cik, yr)
            extracted = None
            if full:
                extracted = extract_section(full, sec)
            if extracted and len(extracted.split()) >= 500:
                print(f"done (EDGAR, {len(extracted.split()):,} words)")
                return extracted, "EDGAR"
            print("EDGAR unavailable or section too short. Loading cache...", end=" ", flush=True)
            cached = fetch_cached_text(slug, yr, sec_key)
            if cached and len(cached.split()) >= 10:
                print(f"done (cache, {len(cached.split()):,} words)")
                return cached, "cache"
            print("cache also unavailable.")
            return None, "unavailable"

        texts, sources = {}, {}
        if section == "Both":
            for sec in ["Item 1A (Risk Factors)", "Item 7 (MD&A)"]:
                t, src = _get_section_text(sec, year)
                texts[sec] = t; sources[sec] = src
            combined = " ".join(t for t in texts.values() if t)
            _cp1_text = combined
        else:
            t, src = _get_section_text(section, year)
            texts[section] = t; sources[section] = src
            _cp1_text = t

        if not _cp1_text or len(_cp1_text.split()) < 10:
            print("No usable text retrieved. Check your connection or ask your facilitator.")
            return

        _cp1_done    = True
        _cp1_year    = year
        _cp1_section = section
        session_data.update({"cp1_year": year, "cp1_section": section})

        wc = len(_cp1_text.split())
        print()
        print(f"Firm        : {_assigned_firm}")
        print(f"Year        : {year}")
        print(f"Section(s)  : {section}")
        print(f"Word count  : {wc:,}")
        for sec, src in sources.items():
            print(f"Source      : {sec} -- {src}")
        print()
        print("Preview (first 200 words):")
        print("-" * 60)
        print(" ".join(_cp1_text.split()[:200]))
        print("-" * 60)

_cp1_btn.on_click(_run_cp1)
display(widgets.VBox([_cp1_section_w, _cp1_btn, _cp1_out]))


In [ ]:
#@title Checkpoint 1 -- Feedback (run after clicking Run Checkpoint 1)

validate_checkpoint(["_cp1_done"], "Checkpoint 1")

_FB_1A = """
Item 1A is a strong choice for detecting forward-looking risk language and changes in
disclosed uncertainty. The section carries a regulatory obligation to disclose material
risks, which gives it a degree of comparability across firms and years that the more
discretionary MD&A does not. That said, the obligation to disclose material risks also
creates an incentive to disclose them in the most defensive and legally cautious language
available -- which means boilerplate language that adds little informational content can
be structurally embedded in this section in ways that will affect your pre-processing and
similarity analysis. Dyer, Lang and Stice-Lawrence (2017) document a substantial expansion
in the length of Item 1A disclosures over time; some of that growth reflects genuine new
risk, and some of it is defensive legal padding. Bear this in mind when you interpret your
Fog score in Checkpoint 4.
"""

_FB_1B = """
MD&A gives you the most direct access to management's own framing of performance -- it is
arguably the richest source of impression management in the entire filing. Li (2008) draws
primarily on this section, and Dyer et al. (2017) show it has grown substantially in length
relative to other filing sections over the past two decades, partly because managers have
increasing latitude over what to include. That latitude is precisely what makes it
analytically interesting: a firm under pressure has more room to manoeuvre in Item 7 than
in Item 1A. The limitation is that this discretion also makes cross-firm comparisons noisier
-- you are partly measuring the firm's situation and partly measuring the CFO's
communication style.
"""

_FB_1C = """
Retrieving both sections gives you the most complete picture, but it introduces a
comparability problem that you need to manage actively. Item 1A and Item 7 serve different
communicative functions: one is primarily legal and defensive, the other is primarily
narrative and interpretive. They will score differently on sentiment, readability, and
similarity -- not because the underlying situation differs, but because the register of the
two sections differs by design. If you analyse them jointly as a single text, you are
blending two distinct communicative acts. If you analyse them separately, you double your
interpretation work. Decide now whether you will treat them as one document or two before
you reach Checkpoint 3, and be consistent.
"""

sec = session_data.get("cp1_section", "")
if "1A" in sec and "7" not in sec:
    display(Markdown(f"**Checkpoint 1 feedback -- Item 1A choice:**\n{_FB_1A}"))
elif "7" in sec and "1A" not in sec:
    display(Markdown(f"**Checkpoint 1 feedback -- Item 7 choice:**\n{_FB_1B}"))
elif sec == "Both":
    display(Markdown(f"**Checkpoint 1 feedback -- Both sections:**\n{_FB_1C}"))


---

## Checkpoint 2: Pre-processing Decisions

*Lecture connection: Week 6, Section 6.4*  |  *Estimated time: 10 minutes*

Configure the pre-processing pipeline. The word cloud and comparison reference panel will display automatically after you click Run. The reference panel always shows what the alternative normalisation method would have produced for the top 10 content words, regardless of your choice -- examine it before proceeding to Checkpoint 3.

In [ ]:
#@title Checkpoint 2 -- Pre-processing

validate_checkpoint(["_cp1_done"], "Checkpoint 1")

_cp2_norm_w = widgets.Dropdown(
    description="Normalisation:",
    options=["None", "Porter Stemmer", "Lemmatisation (spaCy en_core_web_md)"],
    value="Lemmatisation (spaCy en_core_web_md)",
    layout=widgets.Layout(width="400px")
)
_cp2_sw_w = widgets.Dropdown(
    description="Stop words:",
    options=["NLTK standard", "L&M-aware custom list", "None"],
    value="L&M-aware custom list",
    layout=widgets.Layout(width="320px")
)
_cp2_nums_w = widgets.RadioButtons(
    description="Remove numbers:",
    options=["Yes", "No"],
    value="Yes",
    layout=widgets.Layout(width="280px")
)
_cp2_lower_w = widgets.RadioButtons(
    description="Lowercase:",
    options=["Yes", "No"],
    value="Yes",
    layout=widgets.Layout(width="260px")
)
_cp2_out = widgets.Output()
_cp2_btn = widgets.Button(description="Run Checkpoint 2", button_style="primary",
                          layout=widgets.Layout(width="190px"))

def _run_cp2(b):
    global _cp2_done, _cp2_tokens, _cp2_norm, _cp2_sw, _cp2_remove_nums, _cp2_lower
    with _cp2_out:
        clear_output(wait=True)
        norm    = _cp2_norm_w.value
        sw      = _cp2_sw_w.value
        rm_nums = _cp2_nums_w.value == "Yes"
        lower   = _cp2_lower_w.value == "Yes"
        text    = _cp1_text

        print("Applying pipeline...", end=" ", flush=True)
        tokens_chosen = apply_pipeline(text, norm, sw, rm_nums, lower)
        print(f"done ({len(tokens_chosen):,} tokens after filtering)")

        raw_tokens = re.findall(r"[a-zA-Z]+", text.lower())
        print(f"Token count before filtering : {len(raw_tokens):,}")
        print(f"Token count after filtering  : {len(tokens_chosen):,}")
        print()

        # Word cloud
        freq = Counter(tokens_chosen)
        if freq:
            wc_obj = WordCloud(width=800, height=350, background_color="white",
                               max_words=50, prefer_horizontal=0.9)
            wc_obj.generate_from_frequencies(freq)
            plt.figure(figsize=(11, 4.5))
            plt.imshow(wc_obj, interpolation="bilinear")
            plt.axis("off")
            plt.title("Top tokens by frequency after pipeline", fontsize=11)
            plt.tight_layout()
            plt.show()

        # Reference panel: top 10 content tokens, Porter vs lemma
        sw_ref = set(stopwords.words("english"))
        raw_content = [t for t in re.findall(r"[a-zA-Z]{4,}", text.lower())
                       if t not in sw_ref]
        top10 = [w for w, _ in Counter(raw_content).most_common(10)]
        porter_forms = [_stemmer.stem(w) for w in top10]
        if _SPACY_OK:
            doc = _nlp(" ".join(top10))
            lemma_forms = [t.lemma_.lower() for t in doc]
        else:
            lemma_forms = top10
        print("Reference panel -- what the alternative normalisation produces:")
        print(f"{"Token":<20} {"Porter Stem":<22} {"SpaCy Lemma":<22}")
        print("-" * 64)
        for orig, ps, lm in zip(top10, porter_forms, lemma_forms):
            print(f"{orig:<20} {ps:<22} {lm:<22}")
        print()

        _cp2_done    = True
        _cp2_tokens  = tokens_chosen
        _cp2_norm    = norm
        _cp2_sw      = sw
        _cp2_remove_nums = _cp2_nums_w.value
        _cp2_lower   = _cp2_lower_w.value
        session_data.update({
            "cp2_normalisation":  norm,
            "cp2_stopwords":      sw,
            "cp2_remove_numbers": _cp2_nums_w.value,
            "cp2_lowercase":      _cp2_lower_w.value
        })
        print("Checkpoint 2 complete. Run the feedback cell below.")

_cp2_btn.on_click(_run_cp2)
display(widgets.VBox([_cp2_norm_w, _cp2_sw_w,
                      widgets.HBox([_cp2_nums_w, _cp2_lower_w]),
                      _cp2_btn, _cp2_out]))


In [ ]:
#@title Checkpoint 2 -- Feedback (run after clicking Run Checkpoint 2)

validate_checkpoint(["_cp2_done"], "Checkpoint 2")

# Feedback for all nine normalisation x stop-word combinations

_FB_NONE_NONE = '''
No normalisation and no stop-word removal is the maximally transparent pipeline:
every token reaches the dictionary lookup exactly as it appears in the filing.
The cost is noise. Function words such as "the", "a", "of", and "in" will dominate
your token counts and dilute your sentiment intensity scores, because neither the
L&M nor the Harvard dictionary contains them. Numbers and ticker symbols will also
survive and inflate the denominator. This configuration is defensible only as a
diagnostic baseline -- it shows what the raw filing contains before any transformation.
'''

_FB_NONE_NLTK = '''
Retaining surface forms while removing common English stop words is a reasonable
baseline, but the NLTK standard list was designed for general-purpose text, not
financial disclosure. It removes modal verbs including "may", "could", "might",
"should", "would", and "will". In the Loughran and McDonald (2011) framework, modals
are the primary carriers of forward-looking uncertainty -- their removal before the
dictionary lookup suppresses your Uncertainty category score. If your assigned firm's
filing contains significant hedging language, this pipeline will undercount it.
'''

_FB_NONE_LM = '''
No normalisation with L&M-aware stop-word removal is a well-considered combination.
The L&M-aware list retains modal verbs and negation terms, preserving the uncertainty
and sentiment signals that matter most for 10-K analysis. The only remaining limitation
is surface form variation: "restructuring", "restructured", and "restructures" are
counted separately and only match your lexicon if it contains all three forms. For the
L&M dictionary this is less of a problem than for Harvard IV-4, because L&M includes a
wider range of inflected forms. Check the matched-term panel in Checkpoint 3.
'''

_FB_PORTER_NONE = '''
The Porter Stemmer unifies inflected forms -- "restructuring", "restructured", and
"restructures" all become "restructur" -- which improves dictionary recall. However,
without stop-word removal, all function words remain and dilute your sentiment scores.
The Porter Stemmer also produces artefacts in financial vocabulary: "earnings" becomes
"earn", "proceedings" becomes "proce", and some technical terms are truncated in ways
that prevent lexicon matching. Before Checkpoint 3, scan the top 50 tokens in your
word cloud and confirm that the stems are semantically interpretable.
'''

_FB_PORTER_NLTK = '''
This is the most common default combination and it is functional. However, it carries
two risks that are particularly relevant in a finance context. First, the Porter Stemmer
can produce artefacts in financial vocabulary -- check your word cloud for any stems that
look broken or semantically odd. Second, the NLTK standard stop-word list removes modal
verbs such as "may", "could", "might", and "should". In the Loughran and McDonald (2011)
framework, these are uncertainty markers -- their removal will suppress your Uncertainty
category score. If your research question involves detecting hedging language or
forward-looking uncertainty, this combination will undercount it systematically.
'''

_FB_PORTER_LM = '''
Combining the Porter Stemmer with the L&M-aware stop-word list retains the recall
advantage of stemming while preserving the modal verbs and negations that carry
uncertainty and sentiment signals in financial text. This is a reasonable choice for
a workshop setting where computational speed matters. The residual risk is stemmer
artefacts in specialised financial vocabulary; the L&M dictionary was built on
lemmatised forms, so some stems may not match exactly. Check the matched-term counts
in Checkpoint 3 and compare them with what you would expect from the filing's content.
'''

_FB_LEMMA_NONE = '''
Lemmatisation without stop-word removal gives you the most linguistically accurate root
forms available -- "restructuring" becomes "restructure", "was" becomes "be" -- but
retains all function words. The result is a large vocabulary dominated by high-frequency
grammatical tokens that carry no weight in either dictionary. Your sentiment intensity
scores will be diluted by this denominator inflation. This configuration is more useful
as a preprocessing step for topic modelling than for dictionary-based sentiment scoring.
Unless you have a specific reason to retain function words, apply a stop-word list.
'''

_FB_LEMMA_NLTK = '''
Lemmatisation using a spaCy medium model produces more accurate root forms than stemming,
but combining it with the NLTK standard stop-word list reintroduces the modal verb problem.
The list removes "may", "could", "might", "should", "would", and "will" before your tokens
reach the L&M dictionary, suppressing Uncertainty category scores. In a dataset where
forward-looking language is analytically important -- and in 10-K Risk Factor sections it
often is -- this suppression is a substantive loss. The upgrade from Porter to Lemmatisation
is worthwhile; the upgrade from NLTK to L&M-aware stop words would complete the improvement.
If you are committed to this combination, note the modal loss in your analyst's note.
'''

_FB_LEMMA_LM = '''
This is the most carefully considered combination for a 10-K analysis pipeline.
Lemmatisation using a medium-sized spaCy model preserves semantically important
distinctions that a stemmer would collapse. The L&M-aware stop-word list retains modal
verbs and other uncertainty markers, which means your Uncertainty category score in
Checkpoint 3 will have better recall. The main limitation is that lemmatisation quality
depends on part-of-speech tagging, which can be unreliable on highly technical financial
prose. Check the reference panel and confirm that the lemmatiser has handled the financial
vocabulary in your filing sensibly before drawing conclusions from the token counts.
'''

_FB_NUMS = '''
Note: retaining numbers inflates your token count with financial figures that carry no
weight in either the L&M dictionary or the Gunning Fog calculation. This artificially
dilutes your sentiment intensity score and may distort the word cloud. Unless you have
a specific reason to preserve numerical tokens, removing them is the cleaner choice.
'''

norm = session_data.get("cp2_normalisation", "")
sw   = session_data.get("cp2_stopwords", "")
rm   = session_data.get("cp2_remove_numbers", "Yes")

_FEEDBACK_MAP = {
    ("None",           "None"):                  ("No normalisation, no stop-word removal:",        _FB_NONE_NONE),
    ("None",           "NLTK standard"):          ("No normalisation + NLTK stop words:",            _FB_NONE_NLTK),
    ("None",           "L&M-aware custom list"):  ("No normalisation + L&M-aware stop words:",       _FB_NONE_LM),
    ("Porter Stemmer", "None"):                   ("Porter Stemmer, no stop-word removal:",          _FB_PORTER_NONE),
    ("Porter Stemmer", "NLTK standard"):          ("Porter Stemmer + NLTK stop words:",              _FB_PORTER_NLTK),
    ("Porter Stemmer", "L&M-aware custom list"):  ("Porter Stemmer + L&M-aware stop words:",        _FB_PORTER_LM),
}
_lemma = "Lemmatisation" in norm
_feedback_entry = _FEEDBACK_MAP.get((norm, sw))
if _feedback_entry is None and _lemma:
    if sw == "None":
        _feedback_entry = ("Lemmatisation, no stop-word removal:",          _FB_LEMMA_NONE)
    elif sw == "NLTK standard":
        _feedback_entry = ("Lemmatisation + NLTK stop words:",              _FB_LEMMA_NLTK)
    elif sw == "L&M-aware custom list":
        _feedback_entry = ("Lemmatisation + L&M-aware stop words:",         _FB_LEMMA_LM)

if _feedback_entry:
    display(Markdown(f"**Checkpoint 2 feedback -- {_feedback_entry[0]}**\n{_feedback_entry[1]}"))
else:
    print("No specific feedback for this combination. Review the reference panel above "
          "and note your rationale in the analyst's note at Checkpoint 6.")

if rm == "No":
    display(Markdown(f"**Additional note:**\n{_FB_NUMS}"))


---

## Checkpoint 3: Sentiment Analysis

*Lecture connection: Week 7, Section 7.1*  |  *Estimated time: 15 minutes*

Select your dictionary and denominator. If you select "Both (comparison mode)", a divergence panel will show which words drive the score difference between Harvard IV-4 and L&M. This is the most informative setting for understanding dictionary choice consequences.

In [ ]:
#@title Checkpoint 3 -- Sentiment Analysis

validate_checkpoint(["_cp2_done"], "Checkpoint 2")

_cp3_dict_w = widgets.Dropdown(
    description="Dictionary:",
    options=["Harvard IV-4 only", "L&M only", "Both (comparison mode)"],
    value="L&M only",
    layout=widgets.Layout(width="340px")
)
_cp3_denom_w = widgets.Dropdown(
    description="Denominator:",
    options=["Total word count", "Total non-stopword count"],
    value="Total word count",
    layout=widgets.Layout(width="340px")
)
_cp3_out = widgets.Output()
_cp3_btn = widgets.Button(description="Run Checkpoint 3", button_style="primary",
                          layout=widgets.Layout(width="190px"))

def _run_cp3(b):
    global _cp3_done, _cp3_lm_net, _cp3_harv_net, _cp3_dict, _cp3_denom
    with _cp3_out:
        clear_output(wait=True)
        dict_choice = _cp3_dict_w.value
        denom_choice = _cp3_denom_w.value
        tokens = _cp2_tokens

        print("Loading dictionaries...", end=" ", flush=True)
        lm_neg, lm_pos, lm_unc = load_lm_dictionary()
        harv_neg = fetch_wordlist(HARV_NEG_URL)
        harv_pos = fetch_wordlist(HARV_POS_URL)
        print("done")

        sw_tokens = apply_pipeline(_cp1_text, _cp2_norm, "NLTK standard",
                                   _cp2_remove_nums == "Yes", _cp2_lower == "Yes")
        denom_toks = sw_tokens if denom_choice == "Total non-stopword count" else None

        results = {}
        if dict_choice in ["L&M only", "Both (comparison mode)"]:
            net, m_neg, m_pos = score_sentiment(tokens, lm_neg, lm_pos, denom_toks)
            results["L&M"] = {"net": net, "neg": m_neg, "pos": m_pos}
            _cp3_lm_net = net
        if dict_choice in ["Harvard IV-4 only", "Both (comparison mode)"]:
            net_h, h_neg, h_pos = score_sentiment(tokens, harv_neg, harv_pos, denom_toks)
            results["Harvard"] = {"net": net_h, "neg": h_neg, "pos": h_pos}
            _cp3_harv_net = net_h

        # Display results
        for name, res in results.items():
            net_val = res["net"]
            sign    = "+" if net_val >= 0 else ""
            print(f"{name} net sentiment ({denom_choice}): {sign}{net_val:.4f}")
            print(f"  Positive matches: {len(res['pos']):,} | "
                  f"Top 10: {[w for w,_ in Counter(res['pos']).most_common(10)]}")
            print(f"  Negative matches: {len(res['neg']):,} | "
                  f"Top 10: {[w for w,_ in Counter(res['neg']).most_common(10)]}")
            print()

        # Divergence panel (Both mode)
        if dict_choice == "Both (comparison mode)" and "L&M" in results and "Harvard" in results:
            lm_neg_set  = set(results["L&M"]["neg"])
            h_neg_set   = set(results["Harvard"]["neg"])
            divergent   = sorted(Counter(results["Harvard"]["neg"]).items(),
                                 key=lambda x: -x[1])[:20]
            lm_only     = sorted(Counter(results["L&M"]["neg"]).items(),
                                 key=lambda x: -x[1])[:20]
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
            # Harvard negatives not in L&M (false positives)
            h_fp = [(w, c) for w, c in Counter(results["Harvard"]["neg"]).most_common(12)
                    if w not in lm_neg]
            if h_fp:
                ws, cs = zip(*h_fp)
                ax1.barh(range(len(ws)), cs, color="#f5a623")
                ax1.set_yticks(range(len(ws)))
                ax1.set_yticklabels(ws, fontsize=8)
                ax1.invert_yaxis()
                ax1.set_title("Harvard negatives not in L&M\n(likely false positives)", fontsize=9)
            # L&M negatives (genuine)
            lm_top = Counter(results["L&M"]["neg"]).most_common(12)
            if lm_top:
                ws2, cs2 = zip(*lm_top)
                ax2.barh(range(len(ws2)), cs2, color="#e94560")
                ax2.set_yticks(range(len(ws2)))
                ax2.set_yticklabels(ws2, fontsize=8)
                ax2.invert_yaxis()
                ax2.set_title("L&M negative matches\n(genuine financial negatives)", fontsize=9)
            fig.suptitle(f"Divergence panel: {_assigned_firm} {_cp1_year}", fontsize=11)
            plt.tight_layout()
            plt.show()

        _cp3_done  = True
        _cp3_dict  = dict_choice
        _cp3_denom = denom_choice
        if "_cp3_lm_net" not in globals():   globals()["_cp3_lm_net"]   = None
        if "_cp3_harv_net" not in globals(): globals()["_cp3_harv_net"] = None
        session_data.update({
            "cp3_dictionary":  dict_choice,
            "cp3_denominator": denom_choice,
            "cp3_lm_net":      globals().get("_cp3_lm_net"),
            "cp3_harv_net":    globals().get("_cp3_harv_net"),
        })
        print("Checkpoint 3 complete. Run the feedback cell below.")

_cp3_btn.on_click(_run_cp3)
display(widgets.VBox([_cp3_dict_w, _cp3_denom_w, _cp3_btn, _cp3_out]))


In [ ]:
#@title Checkpoint 3 -- Feedback (run after clicking Run Checkpoint 3)

validate_checkpoint(["_cp3_done"], "Checkpoint 3")

_FB_3_HARV = """
The Harvard IV-4 dictionary was constructed for general psychological research and
performs poorly in a financial disclosure context. Loughran and McDonald (2011) showed
that roughly 75% of the negative words it identifies in 10-K filings are not negative
in a finance context -- canonical examples include 'liability', 'tax', 'foreign', and
'crude'. Your net sentiment score will therefore be systematically more negative than
the text actually warrants, and this bias is not uniform across firms or industries:
a bank or an energy company will be penalised more heavily than a technology firm
simply because of the vocabulary of their industry. This is not a minor calibration
issue -- it is a structural bias that would substantially undermine the validity of
any cross-sectional analysis built on this score.
"""

_FB_3_LM = """
This is the standard choice for 10-K analysis and it is well-justified by Loughran
and McDonald (2011). The dictionary was constructed from 10-K filings specifically,
so it captures the register of formal annual disclosures with considerably better
precision than general-purpose alternatives. One limitation worth keeping in mind:
the L&M master dictionary is updated periodically and the version you are using should
be documented in any research output. A second limitation is that the dictionary was
constructed primarily from US-listed firms; its coverage of non-US reporting conventions
or non-US English variants is less thoroughly validated.
"""

_FB_3_BOTH = """
Running both dictionaries in comparison mode is the most analytically informative
choice here, and it is good research practice. If the two dictionaries agree on the
dominant tone and the score magnitude is broadly similar, you can report this as
informal evidence of robustness. If they diverge substantially, you need to understand
the mechanism before you can draw any conclusion -- look at the divergence panel and
identify the specific words that are driving the difference. In most 10-K analyses,
the Harvard IV-4 score will be more negative than the L&M score, and the gap tends
to be largest for firms in financial services, energy, and manufacturing. Whether your
firm shows this expected pattern is itself informative.
"""

_FB_3_DENOM = """
Scaling by total word count is the approach used in Loughran and McDonald (2011) and
it is the most widely cited baseline, which makes your results comparable to the
published literature. The limitation is that longer documents produce lower sentiment
intensity scores purely as a function of length, even if the absolute number of
sentiment words is higher. This matters most when you are comparing across firms with
very different filing lengths, or across years for a firm whose disclosure length
changed substantially -- which, for several of the firms in this session, it did.
"""

d = session_data.get("cp3_dictionary", "")
if d == "Harvard IV-4 only":
    display(Markdown(f"**Checkpoint 3 feedback -- Harvard IV-4 only:**\n{_FB_3_HARV}"))
elif d == "L&M only":
    display(Markdown(f"**Checkpoint 3 feedback -- L&M only:**\n{_FB_3_LM}"))
elif "Both" in d:
    display(Markdown(f"**Checkpoint 3 feedback -- Both dictionaries:**\n{_FB_3_BOTH}"))

if session_data.get("cp3_denominator") == "Total word count":
    display(Markdown(f"**Note on denominator choice:**\n{_FB_3_DENOM}"))


---

## Checkpoint 4: Readability Analysis

*Lecture connection: Week 7, Section 7.2*  |  *Estimated time: 10 minutes*

Compute the Gunning Fog Index for your filing. The Loughran and McDonald (2014) benchmark for 10-K filings is approximately 19--21. A financial context panel shows net income and operating margin for your firm across the available years. If the Fog score increased in a year where the financial metrics deteriorated, the notebook will flag this as a neutral observation -- your interpretation frame determines what to make of it.

In [ ]:
#@title Checkpoint 4 -- Readability Analysis

validate_checkpoint(["_cp3_done"], "Checkpoint 3")

_cp4_years_w = widgets.Dropdown(
    description="Year mode:",
    options=["Single year only", "Compare all available years"],
    value="Compare all available years",
    layout=widgets.Layout(width="340px")
)
_cp4_frame_w = widgets.RadioButtons(
    description="Interpretation frame:",
    options=[
        "Li (2008): complexity signals obfuscation",
        "Bushee et al. (2018): complexity reflects legitimate subject matter",
        "Not yet determined"
    ],
    value="Not yet determined",
    layout=widgets.Layout(width="560px")
)
_cp4_out = widgets.Output()
_cp4_btn = widgets.Button(description="Run Checkpoint 4", button_style="primary",
                          layout=widgets.Layout(width="190px"))

def _run_cp4(b):
    global _cp4_done, _cp4_fog, _cp4_frame
    with _cp4_out:
        clear_output(wait=True)
        year_mode = _cp4_years_w.value
        frame     = _cp4_frame_w.value

        # Financial context panel
        try:
            fin_df = pd.read_csv(FINANCIAL_URL)
            firm_fin = fin_df[fin_df["Firm"] == _assigned_firm].copy()
            print("Financial context:")
            print(firm_fin[["Firm","Year","NetIncome_M_USD","OperatingMargin_Pct"]]
                  .to_string(index=False))
            print()
        except Exception as e:
            print(f"Financial context load failed: {e}")
            firm_fin = pd.DataFrame()

        # Compute Fog scores
        fog_results = {}
        if year_mode == "Single year only":
            years_to_run = [_cp1_year]
            texts_to_run = {_cp1_year: _cp1_text}
        else:
            years_to_run = _assigned_years
            texts_to_run = {_cp1_year: _cp1_text}
            sec_key = "item_1a" if "1A" in _cp1_section else "item_7"
            for yr in _assigned_years:
                if yr != _cp1_year:
                    t = fetch_cached_text(_assigned_slug, yr, sec_key)
                    if not t:
                        t = fetch_filing_text(_assigned_cik, yr) or ""
                        if t:
                            t = extract_section(t, _cp1_section) or t[:80000]
                    texts_to_run[yr] = t or ""

        for yr in sorted(texts_to_run.keys()):
            txt = texts_to_run[yr]
            if not txt or len(txt.split()) < 50:
                fog_results[yr] = None
                continue
            fog_results[yr] = textstat.gunning_fog(txt)

        print("Gunning Fog Index results:")
        print(f"  (Loughran and McDonald (2014) benchmark for 10-Ks: approx. 19-21)")
        for yr in sorted(fog_results.keys()):
            val = fog_results[yr]
            print(f"  {yr}: {val:.2f}" if val is not None else f"  {yr}: unavailable")
        print()

        # Automatic obfuscation flag
        yrs = sorted(fog_results.keys())
        flagged = []
        for i in range(1, len(yrs)):
            y0, y1 = yrs[i-1], yrs[i]
            if fog_results.get(y0) and fog_results.get(y1):
                fog_inc = fog_results[y1] > fog_results[y0]
                if not firm_fin.empty:
                    row0 = firm_fin[firm_fin["Year"]==y0]
                    row1 = firm_fin[firm_fin["Year"]==y1]
                    if not row0.empty and not row1.empty:
                        ni0 = row0["NetIncome_M_USD"].values[0]
                        ni1 = row1["NetIncome_M_USD"].values[0]
                        if fog_inc and ni1 < ni0:
                            flagged.append((y0, y1))
        if flagged:
            for y0, y1 in flagged:
                print(f"  Observation: Fog score increased from {y0} to {y1} in a year "
                      f"where net income declined. This is a neutral observation "
                      f"-- see your interpretation frame for how to read this.")

        # Multi-year line chart
        if year_mode == "Compare all available years" and len(fog_results) > 1:
            valid = {yr: v for yr, v in fog_results.items() if v is not None}
            if valid:
                fig, ax = plt.subplots(figsize=(9, 4))
                ax.plot(list(valid.keys()), list(valid.values()),
                        marker="o", linewidth=2, color="#7c3aed")
                ax.axhspan(19, 21, alpha=0.12, color="grey",
                           label="L&M (2014) benchmark (19-21)")
                for yr, v in valid.items():
                    ax.annotate(f"{v:.1f}", (yr, v), textcoords="offset points",
                                xytext=(0, 10), ha="center", fontsize=9)
                ax.set_xlabel("Filing year")
                ax.set_ylabel("Gunning Fog Index")
                ax.set_title(f"Gunning Fog Index -- {_assigned_firm}")
                ax.legend(fontsize=8)
                plt.tight_layout()
                plt.show()

        _cp4_done  = True
        _cp4_fog   = fog_results.get(_cp1_year)
        _cp4_frame = frame
        session_data.update({
            "cp4_year_mode": year_mode,
            "cp4_frame":     frame,
            "cp4_fog":       _cp4_fog,
        })
        print("Checkpoint 4 complete. Run the feedback cell below.")

_cp4_btn.on_click(_run_cp4)
display(widgets.VBox([_cp4_years_w, _cp4_frame_w, _cp4_btn, _cp4_out]))


In [ ]:
#@title Checkpoint 4 -- Feedback (run after clicking Run Checkpoint 4)

validate_checkpoint(["_cp4_done"], "Checkpoint 4")

_FB_4_LI = """
The obfuscation hypothesis is well established and empirically supported: Li (2008)
found a significant negative association between annual report readability and both
current earnings levels and earnings persistence. The intuition is that managers
facing poor results have an incentive to make the narrative section of their filing
harder to read, either by increasing sentence length, introducing more complex
vocabulary, or simply adding more text. However, Bushee, Gow and Taylor (2018)
demonstrate that operational complexity is a significant confound: firms operating
across multiple business segments in multiple regulatory jurisdictions will produce
inherently longer and denser disclosures regardless of managerial intent, simply
because there is more to disclose. Before concluding obfuscation, ask whether your
firm became operationally more complex in the period you are examining. For Boeing
teams: was the increase in complexity after 2018 a function of the 737 MAX response,
or was it pre-existing? For SVB teams: did the bank's complexity genuinely grow, or
is the language getting harder to parse in specific sections?
"""

_FB_4_BUSHEE = """
Bushee, Gow and Taylor (2018) make a strong case that controlling for firm complexity
substantially attenuates the obfuscation effect documented by Li (2008). Their argument
is that the correlation between readability and performance partly reflects the fact
that operationally complex firms are harder to manage and harder to write about clearly,
rather than that managers are deliberately obscuring bad news. This is a reasonable
counter-position, but it carries a burden of proof: you need to show that the
complexity in your firm's filing is proportionate to its operational complexity. One
practical test is to look at whether readability changed in sections that are not
directly related to performance -- if the Risk Factors section became harder to read
in the same year as a performance decline, the legitimate complexity explanation is
harder to sustain for that specific section.
"""

_FB_4_UNDEC = """
This is the appropriate position given what a Fog score alone can tell you. The score
measures linguistic complexity; it says nothing about intent, and it cannot distinguish
between a firm that writes badly and a firm that writes strategically. The Li (2008)
and Bushee et al. (2018) findings are both statistically well-supported in large
samples, but your analysis here is a single firm. To move from 'undecided' to a
defensible interpretation, you would typically need to examine whether complexity
increased selectively in the sections discussing weak performance, whether the pattern
is consistent across multiple years, and whether there is any external evidence of
operational complexity change in the same period. Note your reasoning in the analyst's
note in Checkpoint 6 -- stating clearly what additional evidence you would need is a
stronger answer than committing prematurely to one frame.
"""

frame = session_data.get("cp4_frame", "")
if "Li (2008)" in frame:
    display(Markdown(f"**Checkpoint 4 feedback -- Li (2008) frame:**\n{_FB_4_LI}"))
elif "Bushee" in frame:
    display(Markdown(f"**Checkpoint 4 feedback -- Bushee et al. (2018) frame:**\n{_FB_4_BUSHEE}"))
elif "Not yet" in frame:
    display(Markdown(f"**Checkpoint 4 feedback -- Not yet determined:**\n{_FB_4_UNDEC}"))


---

## Checkpoint 5: Cosine Similarity

*Lecture connection: Week 7, Section 7.4*  |  *Estimated time: 8 minutes*

Compare the language of two filing years using cosine similarity. This checkpoint
computes the score under **both** representations -- raw term frequency (TF) and
TF-IDF weighting -- so you can observe directly how much the down-weighting of
common vocabulary affects the result. The Brown and Tucker (2011) sample median for
consecutive MD&A sections is **0.89**. The bar chart shows the top 30 terms by
TF-IDF weight in each document; terms present in one year but not the other are
highlighted in a distinct colour.


In [ ]:
#@title Checkpoint 5 -- Cosine Similarity

validate_checkpoint(["_cp4_done"], "Checkpoint 4")

# Build pair options from assigned years relative to CP1 year
_cp5_pair_opts = []
_idx_t = _assigned_years.index(_cp1_year)
if _idx_t >= 1:
    _cp5_pair_opts.append(f"{_cp1_year} vs. {_assigned_years[_idx_t - 1]}")
if _idx_t >= 2:
    _cp5_pair_opts.append(f"{_cp1_year} vs. {_assigned_years[_idx_t - 2]}")
if not _cp5_pair_opts:
    _cp5_pair_opts = [f"{_assigned_years[-1]} vs. {_assigned_years[-2]}"]

_cp5_pair_w = widgets.Dropdown(
    description="Year pair:",
    options=_cp5_pair_opts,
    layout=widgets.Layout(width="360px")
)
_cp5_out = widgets.Output()
_cp5_btn = widgets.Button(description="Run Checkpoint 5", button_style="primary",
                          layout=widgets.Layout(width="190px"))

def _run_cp5(b):
    global _cp5_done, _cp5_tf_sim, _cp5_tfidf_sim
    with _cp5_out:
        clear_output(wait=True)
        pair_str    = _cp5_pair_w.value
        yrs_in_pair = [int(s.strip()) for s in pair_str.split("vs.")]
        yr_t, yr_comp = yrs_in_pair[0], yrs_in_pair[1]

        text_t  = _cp1_text
        sec_key = "item_1a" if "1A" in _cp1_section else "item_7"
        print(f"Loading comparison year ({yr_comp})...", end=" ", flush=True)
        text_comp = fetch_cached_text(_assigned_slug, yr_comp, sec_key)
        if not text_comp or len(text_comp.split()) < 50:
            full_comp = fetch_filing_text(_assigned_cik, yr_comp)
            if full_comp:
                text_comp = extract_section(full_comp, _cp1_section) or full_comp[:80000]
        if not text_comp or len(text_comp.split()) < 50:
            print("unavailable.")
            print("Could not retrieve comparison year text. Try a different pair or "
                  "check connection.")
            return
        print(f"done ({len(text_comp.split()):,} words)")

        # Compute similarity under both representations
        tf_vec    = CountVectorizer(max_features=5000, min_df=1)
        tfidf_vec = TfidfVectorizer(max_features=5000, min_df=1)
        tf_mat    = tf_vec.fit_transform([text_t, text_comp])
        tfidf_mat = tfidf_vec.fit_transform([text_t, text_comp])
        tf_sim    = round(float(cosine_similarity(tf_mat[0],    tf_mat[1])[0][0]),    4)
        tfidf_sim = round(float(cosine_similarity(tfidf_mat[0], tfidf_mat[1])[0][0]), 4)
        gap       = round(tf_sim - tfidf_sim, 4)

        _cp5_tf_sim    = tf_sim
        _cp5_tfidf_sim = tfidf_sim

        print()
        print(f"  Raw TF cosine similarity  : {tf_sim:.4f}")
        print(f"  TF-IDF cosine similarity  : {tfidf_sim:.4f}")
        print(f"  Gap (TF - TF-IDF)         : {gap:+.4f}")
        print(f"  Brown and Tucker (2011) median: 0.89")
        print()
        if gap > 0.08:
            print("  Large gap: a significant share of the shared vocabulary is boilerplate.\n"
                  "  TF-IDF is the more informative measure of substantive similarity.")
        elif gap < 0.02:
            print("  Small gap: boilerplate is a minor share of shared vocabulary.\n"
                  "  Both scores tell approximately the same story.")
        else:
            print("  Moderate gap: boilerplate contributes to TF but is partially\n"
                  "  down-weighted in TF-IDF.")
        print()

        # Top-30 bar chart using TF-IDF weights
        names = tfidf_vec.get_feature_names_out()
        w_t   = tfidf_mat[0].toarray()[0]
        w_c   = tfidf_mat[1].toarray()[0]
        top30_t = sorted(zip(names, w_t), key=lambda x: -x[1])[:30]
        top30_c = sorted(zip(names, w_c), key=lambda x: -x[1])[:30]
        set_t   = {n for n, _ in top30_t}
        set_c   = {n for n, _ in top30_c}
        only_t  = set_t - set_c
        only_c  = set_c - set_t

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 8))
        def _draw_bar(ax, top_terms, exclusive, title):
            terms, wts = zip(*top_terms) if top_terms else ([], [])
            colors = ["#e94560" if t in exclusive else "#7c3aed" for t in terms]
            ax.barh(range(len(terms)), wts, color=colors, height=0.7)
            ax.set_yticks(range(len(terms)))
            ax.set_yticklabels(terms, fontsize=8)
            ax.invert_yaxis()
            ax.set_title(title, fontsize=9)
            p1 = mpatches.Patch(color="#7c3aed", label="Shared vocabulary")
            p2 = mpatches.Patch(color="#e94560", label="Exclusive to this year")
            ax.legend(handles=[p1, p2], fontsize=7)
        _draw_bar(ax1, top30_t, only_t, f"Top 30 terms (TF-IDF) -- {yr_t}")
        _draw_bar(ax2, top30_c, only_c, f"Top 30 terms (TF-IDF) -- {yr_comp}")
        fig.suptitle(f"TF-IDF term weights: {_assigned_firm} ({pair_str})", fontsize=11)
        plt.tight_layout()
        plt.show()

        _cp5_done = True
        session_data.update({
            "cp5_pair":             pair_str,
            "cp5_tf_similarity":    tf_sim,
            "cp5_tfidf_similarity": tfidf_sim,
        })
        print("Checkpoint 5 complete. Run the feedback cell below.")

_cp5_btn.on_click(_run_cp5)
display(widgets.VBox([_cp5_pair_w, _cp5_btn, _cp5_out]))


In [ ]:
#@title Checkpoint 5 -- Feedback (run after clicking Run Checkpoint 5)

validate_checkpoint(["_cp5_done"], "Checkpoint 5")

_FB_TF_HI = '''
Your raw TF score is at or above the Brown and Tucker (2011) median of 0.89. Under raw
term frequency weighting this should be interpreted cautiously: boilerplate language --
legal disclaimers, standard risk warnings, regulatory formulae -- inflates the shared
vocabulary and keeps the score elevated even when substantive content has changed.
Compare this with your TF-IDF score below: if TF-IDF is notably lower, the gap is the
fingerprint of boilerplate. If both scores are high, the filing-to-filing language is
genuinely stable -- which is a substantive finding in its own right.
'''

_FB_TF_LO = '''
A raw TF score below 0.80 between consecutive annual reports is low enough to be
meaningful even before adjusting for boilerplate. Something substantive has changed in
the vocabulary of these two filings. Because this is a raw TF score, some of the
divergence may reflect changes in legal boilerplate rather than substance -- but at this
level the divergence is almost certainly real. For Boeing 2018-2019 teams, you should
see "MAX", "grounding", and related terms appearing prominently in the 2019 column.
Check the bar chart and identify the terms responsible for the divergence.
'''

_FB_TFIDF_LO = '''
A TF-IDF score below 0.80 is substantively meaningful. Because TF-IDF suppresses
vocabulary common to both documents, a low score here indicates that the distinctive
language of the two filings is genuinely different -- this is not an artefact of
boilerplate shifting. Examine the bar chart: are the divergent terms concentrated in a
particular topic area, such as risk, liquidity, or strategy? Brown and Tucker (2011)
found that low MD&A similarity is associated with larger subsequent earnings surprises.
Does the year of lowest similarity correspond to what you know about your firm's
trajectory?
'''

_FB_TFIDF_HI = '''
A TF-IDF similarity above 0.85 is a stronger signal of low incremental disclosure than
a high raw TF score, precisely because TF-IDF has already removed common boilerplate.
What remains at high similarity is firm-specific vocabulary that has persisted across
the two years. This could mean the firm's situation is genuinely stable, or that
management is recycling prior-year language to describe a changed situation. The bar
chart will help you distinguish: if the high-weight shared terms describe the firm's
core operations, stability is more plausible. If they describe a risk that materialised,
consider whether the language was updated adequately.
'''

_FB_GAP_LARGE = '''
The gap between your TF and TF-IDF scores is large (above 0.08). This shows that a
substantial portion of the vocabulary shared between the two filings consists of common
language that TF-IDF down-weights -- legal boilerplate, formulaic risk disclosures,
standard governance language. The TF-IDF score is the more reliable indicator of how
much the substantive, firm-specific content of the filing changed. Use the TF-IDF
figure as your primary evidence in the analyst's note.
'''

tf_sim    = session_data.get("cp5_tf_similarity",    0.0) or 0.0
tfidf_sim = session_data.get("cp5_tfidf_similarity", 0.0) or 0.0
gap       = tf_sim - tfidf_sim

display(Markdown(
    f"**Checkpoint 5 results:**  "
    f"TF = {tf_sim:.4f}  |  TF-IDF = {tfidf_sim:.4f}  |  Gap = {gap:+.4f}"
))

if tf_sim >= 0.89:
    display(Markdown(f"**Raw TF score ({tf_sim:.4f} >= 0.89):**\n{_FB_TF_HI}"))
elif tf_sim < 0.80:
    display(Markdown(f"**Raw TF score ({tf_sim:.4f} < 0.80):**\n{_FB_TF_LO}"))

if tfidf_sim < 0.80:
    display(Markdown(f"**TF-IDF score ({tfidf_sim:.4f} < 0.80):**\n{_FB_TFIDF_LO}"))
elif tfidf_sim >= 0.85:
    display(Markdown(f"**TF-IDF score ({tfidf_sim:.4f} >= 0.85):**\n{_FB_TFIDF_HI}"))

if gap > 0.08:
    display(Markdown(f"**Boilerplate gap ({gap:+.4f} > 0.08):**\n{_FB_GAP_LARGE}"))

if 0.80 <= tf_sim < 0.89 and 0.80 <= tfidf_sim < 0.85:
    print(f"Both scores in mid-range. Compare the bar chart carefully for the terms "
          f"driving shared and divergent vocabulary before forming a conclusion.")


---

## Checkpoint 6: The Analyst's Note and Submission

*Estimated time: 5 minutes*

Write a short analyst note (maximum 150 words) and submit. Submission writes your team's results and decisions to the shared session record. There is no automated feedback on the note -- it is the material for class discussion.

In [ ]:
#@title Checkpoint 6 -- The Analyst's Note and Submission

validate_checkpoint(["_cp5_done"], "Checkpoint 5")

_CP6_PROMPT = (
    "Write a short analyst note (maximum 150 words) for your research desk summarising "
    "what your textual analysis reveals about {firm}. Address: (1) what the sentiment "
    "signal shows and whether it is consistent with the firm's financial trajectory; "
    "(2) what the readability evidence suggests and which interpretive frame you favour; "
    "(3) what the cosine similarity score implies about the informational content of the "
    "filing. State clearly what you would and would not conclude from this evidence alone."
).format(firm=_assigned_firm)

_cp6_label = widgets.HTML(f'<p style="font-size:13px;max-width:720px">{_CP6_PROMPT}</p>')
_cp6_note_w = widgets.Textarea(
    placeholder="Type your analyst note here (max 150 words)...",
    layout=widgets.Layout(width="720px", height="160px")
)
_cp6_count_out = widgets.Output()
_cp6_out       = widgets.Output()
_cp6_btn       = widgets.Button(description="Submit", button_style="success",
                                layout=widgets.Layout(width="140px"))
_cp6_retry_btn = widgets.Button(description="Retry submission", button_style="warning",
                                layout=widgets.Layout(width="180px", display="none"))

def _update_count(change):
    with _cp6_count_out:
        clear_output(wait=True)
        wc = len(_cp6_note_w.value.split())
        print(f"Word count: {wc}/150")

_cp6_note_w.observe(_update_count, names="value")

def _post_to_endpoint(payload):
    """POST payload to Apps Script endpoint. Returns (success, message)."""
    if not APPS_SCRIPT_URL or "[PASTE" in APPS_SCRIPT_URL:
        return False, "not_configured"
    try:
        resp = requests.post(
            APPS_SCRIPT_URL,
            json=payload,
            timeout=20,
            headers={"Content-Type": "application/json"},
        )
        result = resp.json()
        if result.get("status") == "success":
            return True, "success"
        return False, result.get("message", "server_error")
    except Exception as _ex:
        return False, str(_ex)

def _run_submission(payload):
    """Attempt submission and update UI. Returns True on success."""
    ok, msg = _post_to_endpoint(payload)
    if ok:
        print("Submitted successfully. Your results have been saved.")
        _cp6_retry_btn.layout.display = "none"
        return True
    if msg == "not_configured":
        print("The submission endpoint has not been configured.")
        print("Please let your instructor know before the end of the session.")
    else:
        print("Submission failed -- this is likely a temporary network issue.")
        print("Please click Retry below. If it keeps failing, let your instructor know.")
        print(f"(Technical detail for instructor: {msg})")
    _cp6_retry_btn.layout.display = ""
    return False

def _run_cp6(b):
    global _cp6_payload
    with _cp6_out:
        clear_output(wait=True)
        _cp6_retry_btn.layout.display = "none"
        note = _cp6_note_w.value.strip()
        wc   = len(note.split())
        if not note:
            print("Please write your analyst note before submitting.")
            return
        if wc > 150:
            print(f"Note is {wc} words. Please reduce to 150 words or fewer.")
            return
        session_data["cp6_analyst_note"] = note
        _cp6_payload = {
            "student_id":         f"{session_data.get('team_name', 'unknown')}_{int(time.time())}",
            "team_name":          str(session_data.get("team_name", "")),
            "firm":               str(session_data.get("firm", "")),
            "year":               str(session_data.get("cp1_year", "")),
            "section":            str(session_data.get("cp1_section", "")),
            "normalisation":      str(session_data.get("cp2_normalisation", "")),
            "stopwords":          str(session_data.get("cp2_stopwords", "")),
            "remove_numbers":     str(session_data.get("cp2_remove_numbers", "")),
            "lowercase":          str(session_data.get("cp2_lowercase", "")),
            "dictionary":         str(session_data.get("cp3_dictionary", "")),
            "net_sentiment_lm":   str(session_data.get("cp3_lm_net", "")),
            "net_sentiment_hiv4": str(session_data.get("cp3_harv_net", "")),
            "fog_score":          str(session_data.get("cp4_fog", "")),
            "readability_frame":  str(session_data.get("cp4_frame", "")),
            "cosine_similarity":  str(session_data.get("cp5_tfidf_similarity", "")),
            "tf_or_tfidf":        str(session_data.get("cp5_tf_similarity", "")),
            "comparison_pair":    str(session_data.get("cp5_pair", "")),
            "analyst_note":       str(session_data.get("cp6_analyst_note", "")),
            "timestamp":          time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
        }
        _run_submission(_cp6_payload)

def _retry_cp6(b):
    with _cp6_out:
        clear_output(wait=True)
        print("Retrying submission...")
        _run_submission(_cp6_payload)

_cp6_btn.on_click(_run_cp6)
_cp6_retry_btn.on_click(_retry_cp6)
display(widgets.VBox([_cp6_label, _cp6_note_w, _cp6_count_out,
                      _cp6_btn, _cp6_retry_btn, _cp6_out]))


---

## Sharing Link for Students

Open this link in any browser to launch the workshop notebook in Google Colab:

https://colab.research.google.com/github/iamjamesbowden/AG952/blob/main/materials/week07/notebooks/AG952_Week07_Workshop3_StudentNotebook.ipynb

Students do not need to install anything. They need a Google account.
